# H7 · Qwen3-4B (4-bit + LoRA) as the verdict model (Himasri)

The professor asked about a quantized Qwen3-4B. This fine-tunes it (QLoRA) to read a claim + its top-3 snippets and answer supported / contradicted / unverified, in two versions:
- **orig**: balanced MultiFC train originals (4,500 examples)
- **twins**: the same + 1,800 train twins

Both are scored on MultiFC test + twins and AVeriTeC test with the shared metrics, then compared with our MLP head and the claim-only baseline (paired bootstrap).

**Before you start:** Runtime → **T4 GPU** (L4 / A100 if available: about twice as fast). Colab secret `GH_TOKEN`. Time on a T4: ~10 min setup + ~40 min per version.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!git checkout -q himasri/H7-qwen4b
!pip install -q peft bitsandbytes accelerate
from google.colab import userdata
!ls features/relevance_en.parquet features/relevance_copies_en.parquet data/copies/en

## 2. 10-row check (Qwen3-4B 4-bit, 3 training steps; output in /content/test10)

In [ ]:
!python -u -m src.verdict.llm_verdict --variant twins --n-per-class 4 --n-twin-per-type 2 --limit-test 10 --max-steps 3 --out-dir /content/test10 --models-dir /content/test10/models 2>&1 | grep -E "examples|step|llm4b|Error|Traceback"

## 3. Version `orig` (originals only)

In [ ]:
!python -u -m src.verdict.llm_verdict --variant orig 2>&1 | grep -E "examples|step|llm4b|Error|Traceback"

## 4. Version `twins` (originals + twins)

In [ ]:
!python -u -m src.verdict.llm_verdict --variant twins 2>&1 | grep -E "examples|step|llm4b|Error|Traceback"

## 5. Summary + bootstrap vs our MLP head and the claim-only baseline (paste this into the chat with Claude)

In [ ]:
import json
for v in ('orig', 'twins'):
    m = json.load(open(f'results/H7/qwen4b/metrics_llm4b_{v}_s42.json'))
    mf, av = m['multifc'], m['averitec']
    print(f"llm4b_{v}: MultiFC macro-F1 {mf['macro_f1']:.3f} AUROC {mf['auroc']:.3f} ECE {mf['ece']:.3f} | "
          f"ESR {mf['esr']:.3f} unrelated {mf['unrelated_rate']:.3f} paraphrase {mf['paraphrase_consistency']:.3f} | "
          f"AVeriTeC macro-F1 {av['macro_f1']:.3f} | train {m['train']['train_seconds']}s")
GOLD = 'data/processed/en/multifc_test.jsonl data/copies/en/test.jsonl'
!python -m src.eval.compare --a results/H7/qwen4b/preds_llm4b_twins_s42.jsonl results/H7/qwen4b/preds_llm4b_twins_s42.jsonl results/H7/qwen4b/preds_llm4b_twins_s42.jsonl --b results/H5/preds_twin_s42.jsonl results/F4/preds_claim_only.jsonl results/H7/qwen4b/preds_llm4b_orig_s42.jsonl --gold {GOLD} --out results/H7/qwen4b/bootstrap_llm4b.json | grep -E "vs|macro_f1|esr|unrelated|paraphrase"

## 6. Push to `himasri/H7-qwen4b` (fill in your GitHub email)

In [ ]:
GIT_NAME = 'Himasri'
GIT_EMAIL = 'YOUR_GITHUB_EMAIL'
token = userdata.get('GH_TOKEN')
!git config user.name "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
!git add results/H7/qwen4b models/H7
!git status --short | head
!git commit -q -m "[H7] Qwen3-4B 4-bit LoRA verdict model: orig and twins versions, predictions, metrics, bootstrap"
!git push -q https://{token}@github.com/himaallu/Group-Project.git himasri/H7-qwen4b && echo pushed

When it says `pushed`: Runtime → Disconnect and delete runtime.